In [ ]:
pip install transformers torch sentencepiece

In [ ]:
import re
import nltk
import string
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from transformers import pipeline
import matplotlib.pyplot as plt
from textblob import TextBlob
from sklearn.pipeline import Pipeline
from nltk.corpus import stopwords as nltk_stopwords
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import confusion_matrix as cm
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from wordcloud import WordCloud, STOPWORDS
from sklearn.preprocessing import LabelEncoder
from nltk.stem import PorterStemmer
nltk.download('stopwords')
nltk.download('vader_lexicon')
nltk.download('wordnet')
from nltk.corpus import stopwords
import pickle
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('final_merged_flipkart_reviews.csv')

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df['Overall Rating'].value_counts()

In [ ]:
df = df[df['Overall Rating'] != 'No Rating']

In [ ]:
df['Overall Rating'].value_counts()

In [ ]:
plt.figure(figsize=(8, 6))
sns.countplot(x='Overall Rating', data=df)
plt.title('Distribution of Overall Ratings')
plt.xlabel('Overall Rating')
plt.ylabel('Number of Reviews')
plt.show()

In [ ]:
sentiment_model = pipeline("text-classification", model="cardiffnlp/twitter-roberta-base-sentiment")

In [ ]:
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet

lemmatizer = WordNetLemmatizer()

def cleanReviews(text):
    """Cleans the input text and applies stemming."""
    if not isinstance(text, str) or not text.strip():
        return ""

    # Remove emojis
    emoji_pattern = re.compile("["
                               u"\U0001F600-\U0001F64F"  # emoticons
                               u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                               u"\U0001F680-\U0001F6FF"  # transport & map symbols
                               u"\U0001F1E0-\U0001F1FF"  # flags
                               u"\U00002702-\U000027B0"
                               u"\U000024C2-\U0001F251"
                               "]+", flags=re.UNICODE)
    text = emoji_pattern.sub(r'', text)

    # Convert to lowercase
    text = text.lower()

    # Remove mentions, hashtags, URLs, and special characters
    text = re.sub(r'@[A-Za-z0-9_]+', '', text)  # Remove @mentions
    text = re.sub(r'#', '', text)  # Remove hashtags
    text = re.sub(r'https?://\S+|www\.\S+', '', text)  # Remove URLs
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)  # Remove special characters & punctuation
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra spaces

    # Apply stemming
    text = ' '.join([lemmatizer.lemmatize(word) for word in text.split()])

    return text

In [ ]:
df["cleanedReviews"] = df["Review"].apply(cleanReviews)

In [ ]:
df.head()

In [ ]:
text = " ".join(df['cleanedReviews'])

stopwords = set(STOPWORDS)

wordcloud = WordCloud(stopwords=stopwords,
                      background_color="black",
                      colormap="cool",
                      width=1200,
                      height=800,
                      max_words=200,
                      contour_color='white',
                      contour_width=2).generate(text)

# Display the WordCloud
plt.figure(figsize=(12, 8))
plt.imshow(wordcloud, interpolation='antialiased')
plt.axis("off")
plt.show()

In [ ]:
def analyze_sentiment(text):
    cleaned_text = cleanReviews(text)
    result = sentiment_model(cleaned_text)[0]

    label = result['label']
    score = result['score']

    if label == "LABEL_2":
        sentiment = "Positive"
    elif label == "LABEL_0":
        sentiment = "Negative"
    else:
        sentiment = "Neutral"

    return sentiment, score  # Return as a tuple instead of Series

df[['Sentiment', 'Sentiment Score']] = df['cleanedReviews'].apply(analyze_sentiment).apply(pd.Series)

In [ ]:
reviews = [
    "It's not a bad product",
    "It's not very amazing",
    "I absolutely love this!",
    "This is the worst purchase ever!",
    "I don't hate it, but I don't love it either.",
    "Not great, not terrible.",
    "Wow, this exceeded my expectations!",
    "This was supposed to be great, but it isn't.",
    "I was expecting better.",
    "Could be worse, but also could be better.",
]

for review in reviews:
    print(f"Review: {review} -> Sentiment: {analyze_sentiment(review)}")

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
plt.figure(figsize=(7, 5))
sns.countplot(x="Sentiment", data=df, palette="rocket")
plt.title("Sentiment Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Count")
plt.show()

In [ ]:
plt.figure(figsize=(6, 6))
df['Sentiment'].value_counts().plot.pie(autopct='%1.1f%%', colors=['lightcoral', 'lightblue', 'lightgreen'])
plt.title("Sentiment Proportion")
plt.ylabel("")
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
sns.boxplot(x="Sentiment", y="Sentiment Score", data=df, palette="coolwarm")
plt.title("Sentiment Score Distribution")
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
sns.histplot(df['Sentiment Score'], bins=20, kde=True, color='purple')
plt.title("Sentiment Score Distribution")
plt.xlabel("Sentiment Score")
plt.ylabel("Density")
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
sns.violinplot(x="Sentiment", y="Sentiment Score", data=df, palette="coolwarm")
plt.title("Sentiment Score Distribution by Sentiment Type")
plt.show()

In [ ]:
min_count = df['Sentiment'].value_counts().min()
df = df.groupby('Sentiment').apply(lambda x: x.sample(min_count)).reset_index(drop=True)

In [ ]:
sentiment_counts = df['Sentiment'].value_counts()
plt.figure(figsize=(8, 6))
sns.barplot(x=sentiment_counts.index, y=sentiment_counts.values, palette='viridis')
plt.title('Balanced Sentiment Analysis Results')
plt.xlabel('Sentiment')
plt.ylabel('Number of Reviews')
plt.show()

In [ ]:
le = LabelEncoder()
df['Sentiment_enc'] = le.fit_transform(df['Sentiment'])

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
X = df['cleanedReviews']
Y = df['Sentiment_enc']

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [ ]:
vectorizer = TfidfVectorizer(stop_words="english",max_features=5000)
X_train_vect = vectorizer.fit_transform(X_train)
X_test_vect = vectorizer.transform(X_test)

In [ ]:
model = MultinomialNB()
model.fit(X_train_vect, Y_train)

In [ ]:
Y_pred = model.predict(X_test_vect)
Y_pred

In [ ]:
print("Accuracy Score:", accuracy_score(Y_test, Y_pred))
print("Classification Report:\n", classification_report(Y_test, Y_pred))
print("Confusion Matrix:")
sns.heatmap(cm(Y_test, Y_pred), annot=True, fmt='d', cmap='Blues')
plt.show()

In [ ]:
def predict_sentiment(review):
    review = cleanReviews(review)
    review_vec = vectorizer.transform([review])
    prediction = model.predict(review_vec)[0]
    return le.inverse_transform([prediction])[0]

In [ ]:
sample_reviews = [
    "The product is amazing! I love it!",
    "Worst product ever. Waste of money!",
    "It's okay, nothing special.",
    "It is not a bad product.",
    "Not great, but not terrible either."
]

for review in sample_reviews:
    print(f"Review: {review} -> Sentiment: {predict_sentiment(review)}")

In [ ]:
with open("vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

with open("sentimental_model.pkl", "wb") as f:
    pickle.dump(model, f)